# Wave Equation with NumPy

Evolve the one-dimensional wave equation directly in NumPy before asking NRPy to generate C.

Navigation: [Index](../index.ipynb) | Previous: [Reference Metrics](../1-intro/reference_metric.ipynb) | Next: [Method of Lines and Runge-Kutta Methods](method_of_lines_and_rk.ipynb)


## Evolve a Periodic Wave
Ghost zones are extra cells around the domain that make stencil reads near boundaries use the same formula as interior reads. A Courant factor sets the time step as a fraction of the grid spacing.

## Step 1: Choose constants and inspect the first grid spacing

Choose constants and inspect the first grid spacing.

In [ ]:
import numpy as np

final_time = 0.2
courant_factor = 0.25
length = 2.0 * np.pi
npoints = 64
dx = length / npoints
print("dx:", f"{dx:.6e}")


## Step 2: Define periodic boundary handling and the fourth-order second-derivative stencil

Define periodic boundary handling and the fourth-order second-derivative stencil.

In [ ]:
def laplacian_periodic(u, dx):
    padded = np.pad(u, 2, mode="wrap")
    return (
        -padded[0:-4]
        + 16.0 * padded[1:-3]
        - 30.0 * padded[2:-2]
        + 16.0 * padded[3:-1]
        - padded[4:]
    ) / (12.0 * dx * dx)


x = dx * np.arange(npoints)
u = np.sin(x)
print("laplacian sample:", [f"{value:.6f}" for value in laplacian_periodic(u, dx)[:4]])


## Step 3: Define one RK4 update for the first-order wave system

Define one RK4 update for the first-order wave system.

In [ ]:
def rhs(u, v, dx):
    return v, laplacian_periodic(u, dx)


def rk4_step(u, v, dx, dt):
    k1u, k1v = rhs(u, v, dx)
    k2u, k2v = rhs(u + 0.5 * dt * k1u, v + 0.5 * dt * k1v, dx)
    k3u, k3v = rhs(u + 0.5 * dt * k2u, v + 0.5 * dt * k2v, dx)
    k4u, k4v = rhs(u + dt * k3u, v + dt * k3v, dx)
    u_next = u + dt * (k1u + 2.0 * k2u + 2.0 * k3u + k4u) / 6.0
    v_next = v + dt * (k1v + 2.0 * k2v + 2.0 * k3v + k4v) / 6.0
    return u_next, v_next


print("RK4 helper ready")


## Step 4: Evolve three resolutions and print a convergence table

Evolve three resolutions and print a convergence table.

In [ ]:
def evolve(npoints):
    dx = length / npoints
    dt = courant_factor * dx
    steps = int(np.ceil(final_time / dt))
    dt = final_time / steps
    x = dx * np.arange(npoints)
    u = np.sin(x)
    v = np.zeros_like(u)
    for _ in range(steps):
        u, v = rk4_step(u, v, dx, dt)
    exact_u = np.sin(x) * np.cos(final_time)
    return dx, steps, float(np.max(np.abs(u - exact_u)))


rows = [(npoints, *evolve(npoints)) for npoints in [64, 128, 256]]
print("npoints dx steps max_error convergence_factor")
previous_error = None
for npoints, dx, steps, error in rows:
    factor = "" if previous_error is None else f"{previous_error / error:.3f}"
    print(npoints, f"{dx:.6e}", steps, f"{error:.6e}", factor)
    previous_error = error


## Step 5: Check that refinement improves the error

Check that refinement improves the error.

In [ ]:
coarse_error = rows[0][3]
fine_error = rows[-1][3]
print("coarse error:", f"{coarse_error:.6e}")
print("fine error:", f"{fine_error:.6e}")
if fine_error >= coarse_error:
    raise RuntimeError("Expected the refined grid to reduce the maximum error.")


The diagnostic table shows a stable finite-difference wave update in plain NumPy. NRPy automates the same mathematical pieces when later notebooks replace hand-written loops with generated kernels.


## Continue to Method of Lines
- [Method of Lines and Runge-Kutta](method_of_lines_and_rk.ipynb)
- [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb)
- [Wave Equation and C Code Generation](../3-wave_equation/wave_equation_and_c_codegen.ipynb)
